# Problem statement

In [1]:
a = 10
b =20
a+b

30

#### PCCOR&R

- a
- b

To predict whether a new mail based on its content, can be categorized as spam or not-spam.

### Data processing using panda library

In [3]:
import pandas as pd

In [41]:
# Import the required libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import string
import matplotlib.pyplot as plt

In [2]:
# pip install openpyxl

In [4]:
df = pd.read_csv("E01_spam.csv", encoding='latin-1', names=['Class','Message'])
df.drop(index = 0, inplace=True) 

In [5]:
df

,Class,Message
1,ham,"Go until jurong point, crazy.. Available only ..."
2,ham,Ok lar... Joking wif u oni...
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...
4,ham,U dun say so early hor... U c already then say...
5,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5568,spam,This is the 2nd time we have tried 2 contact u...
5569,ham,Will Ì_ b going to esplanade fr home?
5570,ham,"Pity, * was in mood for that. So...any other s..."
5571,ham,The guy did some bitching but I acted like i'd...


In [6]:
# to view the first record
df.loc[:5]

,Class,Message
1,ham,"Go until jurong point, crazy.. Available only ..."
2,ham,Ok lar... Joking wif u oni...
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...
4,ham,U dun say so early hor... U c already then say...
5,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.iloc[6,1]

'Even my brother is not like to speak with me. They treat me like aids patent.'

In [8]:
# Summary of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 1 to 5572
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Class    5572 non-null   object
 1   Message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [9]:
# create a column to keep the count of the characters present in each record
df['Length'] = df['Message'].astype(str).apply(len)

In [10]:
df['Length']

1       111
2        29
3       155
4        49
5        61
       ... 
5568    161
5569     37
5570     57
5571    125
5572     26
Name: Length, Length: 5572, dtype: int64

In [12]:
# view the dataset with the column 'Length' which contains the number of characters present in each mail
df.head(10)

,Class,Message,Length
1,ham,"Go until jurong point, crazy.. Available only ...",111
2,ham,Ok lar... Joking wif u oni...,29
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155
4,ham,U dun say so early hor... U c already then say...,49
5,ham,"Nah I don't think he goes to usf, he lives aro...",61
6,spam,FreeMsg Hey there darling it's been 3 week's n...,148
7,ham,Even my brother is not like to speak with me. ...,77
8,ham,As per your request 'Melle Melle (Oru Minnamin...,160
9,spam,WINNER!! As a valued network customer you have...,158
10,spam,Had your mobile 11 months or more? U R entitle...,154


In [13]:
df.tail()

,Class,Message,Length
5568,spam,This is the 2nd time we have tried 2 contact u...,161
5569,ham,Will Ì_ b going to esplanade fr home?,37
5570,ham,"Pity, * was in mood for that. So...any other s...",57
5571,ham,The guy did some bitching but I acted like i'd...,125
5572,ham,Rofl. Its true to its name,26


In [9]:
## The mails are categorised into 2 classes ie., spam and ham.
# Let's see the count of each class
df.groupby('Class').count()

,Message,Length
Class,,
ham,4825,4825
spam,747,747


### Data Visualization

In [14]:
df['Length'].describe() # to find the max length of the message.

count    5572.000000
mean       80.118808
std        59.690841
min         2.000000
25%        36.000000
50%        61.000000
75%       121.000000
max       910.000000
Name: Length, dtype: float64

In [15]:
df.describe()

,Length
count,5572.000000
mean,80.118808
std,59.690841
min,2.000000
25%,36.000000
50%,61.000000
75%,121.000000
max,910.000000


In [12]:
df.describe(include="O")

,Class,Message
count,5572,5572
unique,2,5169
top,ham,"Sorry, I'll call later"
freq,4825,30


See what we found, A 910 character long message. Let's use masking to find this message:

In [13]:
df['Length']==910

1       False
2       False
3       False
4       False
5       False
        ...  
5568    False
5569    False
5570    False
5571    False
5572    False
Name: Length, Length: 5572, dtype: bool

In [14]:
# the message that has the max characters
df[df['Length']==910]['Message']

1085    For me the love should start with attraction.i...
Name: Message, dtype: object

In [15]:
df.iloc[1085]

Class                                                    ham
Message    FR'NDSHIP is like a needle of a clock. Though ...
Length                                                   158
Name: 1086, dtype: object

In [16]:
# view the message that has 910 characters in it
df[df['Length']==910]['Message'].iloc[0]

"For me the love should start with attraction.i should feel that I need her every time around me.she should be the first thing which comes in my thoughts.I would start the day and end it with her.she should be there every time I dream.love will be then when my every breath has her name.my life should happen around her.my life will be named to her.I would cry for her.will give all my happiness and take all her sorrows.I will be ready to fight with anyone for her.I will be in love when I will be doing the craziest things for her.love will be when I don't have to proove anyone that my girl is the most beautiful lady on the whole planet.I will always be singing praises for her.love will be when I start up making chicken curry and end up makiing sambar.life will be the most beautiful then.will get every morning and thank god for the day because she is with me.I would like to say a lot..will tell later.."

In [17]:
# View the message that has min characters
df[df['Length']==2]['Message'].iloc[0]

'Ok'

### Text Pre-Processing

In [16]:
# creating an object for the target values
dObject = df['Class'].values
dObject

array(['ham', 'ham', 'spam', ..., 'ham', 'ham', 'ham'], dtype=object)

In [17]:
df['Target'] = df['Class'].map({"spam":0,'ham':1})

In [18]:
df.head()

,Class,Message,Length,Target
1,ham,"Go until jurong point, crazy.. Available only ...",111,1
2,ham,Ok lar... Joking wif u oni...,29,1
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0
4,ham,U dun say so early hor... U c already then say...,49,1
5,ham,"Nah I don't think he goes to usf, he lives aro...",61,1


In [19]:
# Lets assign ham as 1
# df.loc[df['Class']=="ham","Class"] = 1   # Alteranate encloding technique

In [20]:
# Lets assign spam as 0
# data.loc[data['Class']=="spam","Class"] = 0   # Alteranate encloding technique

In [21]:
dObject2=df['Class'].values
dObject2

array(['ham', 'ham', 'spam', ..., 'ham', 'ham', 'ham'], dtype=object)

In [22]:
df.head(8)

,Class,Message,Length,Target
1,ham,"Go until jurong point, crazy.. Available only ...",111,1
2,ham,Ok lar... Joking wif u oni...,29,1
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0
4,ham,U dun say so early hor... U c already then say...,49,1
5,ham,"Nah I don't think he goes to usf, he lives aro...",61,1
6,spam,FreeMsg Hey there darling it's been 3 week's n...,148,0
7,ham,Even my brother is not like to speak with me. ...,77,1
8,ham,As per your request 'Melle Melle (Oru Minnamin...,160,1


First removing punctuation. We can just take advantage of Python's built-in string library to get a quick list of all the possible punctuation:

In [23]:
# the default list of punctuations
import string
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [24]:
# Why is it important to remove punctuation?

"This message is spam" == "This message is spam,"

False

In [25]:
l1=['a','b','c']
" ".join(l1)

'a b c'

In [28]:
l1=['a','b','c']
" ".join(l1)

'a b c'

In [26]:
str1=df['Message'][1]

In [27]:
str1

'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'

In [28]:
import string

In [29]:
l1 = [i for i in str1 if i not in string.punctuation]  #list comprehension
print(l1)
"".join(l1)

['G', 'o', ' ', 'u', 'n', 't', 'i', 'l', ' ', 'j', 'u', 'r', 'o', 'n', 'g', ' ', 'p', 'o', 'i', 'n', 't', ' ', 'c', 'r', 'a', 'z', 'y', ' ', 'A', 'v', 'a', 'i', 'l', 'a', 'b', 'l', 'e', ' ', 'o', 'n', 'l', 'y', ' ', 'i', 'n', ' ', 'b', 'u', 'g', 'i', 's', ' ', 'n', ' ', 'g', 'r', 'e', 'a', 't', ' ', 'w', 'o', 'r', 'l', 'd', ' ', 'l', 'a', ' ', 'e', ' ', 'b', 'u', 'f', 'f', 'e', 't', ' ', 'C', 'i', 'n', 'e', ' ', 't', 'h', 'e', 'r', 'e', ' ', 'g', 'o', 't', ' ', 'a', 'm', 'o', 'r', 'e', ' ', 'w', 'a', 't']


'Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat'

In [30]:
s = "we are datamites"
l = s.split()
print(l)
s1 = " ".join(l)
print(s1)

['we', 'are', 'datamites']
we are datamites


In [31]:
def remove(text):
    out=''
    for i in text:
        if i not in string.punctuation:
            out=out+i
    return out

In [32]:
remove('we #$$$@at datam*^&%^$%$!ite^^^%%$$#$^&*^&%W#Q')

'we at datamiteWQ'

In [33]:
remove(str1)

'Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat'

In [34]:
l1=[]
for i in str1:
    if i not in string.punctuation:
        l1.append(i)

In [35]:
"".join(l1)

'Go until jurong point crazy Available only in bugis n great world la e buffet Cine there got amore wat'

In [36]:
"hello"=="Hello"

False

In [37]:
# Let's remove the punctuation

def remove_punct(text):
    text = "".join([char for char in text if char not in string.punctuation])
    return text.lower()

df['text_clean'] = df['Message'].astype(str).apply(lambda x: remove_punct(x))

df.head()

,Class,Message,Length,Target,text_clean
1,ham,"Go until jurong point, crazy.. Available only ...",111,1,go until jurong point crazy available only in ...
2,ham,Ok lar... Joking wif u oni...,29,1,ok lar joking wif u oni
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,free entry in 2 a wkly comp to win fa cup fina...
4,ham,U dun say so early hor... U c already then say...,49,1,u dun say so early hor u c already then say
5,ham,"Nah I don't think he goes to usf, he lives aro...",61,1,nah i dont think he goes to usf he lives aroun...


__Tokenization__ (process of converting the normal text strings in to a list of tokens(also known as lemmas)).

In [45]:
# original text and cleaned text
df.head(8)

,Class,Message,Length,Target,text_clean
1,ham,"Go until jurong point, crazy.. Available only ...",111,1,go until jurong point crazy available only in ...
2,ham,Ok lar... Joking wif u oni...,29,1,ok lar joking wif u oni
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,free entry in 2 a wkly comp to win fa cup fina...
4,ham,U dun say so early hor... U c already then say...,49,1,u dun say so early hor u c already then say
5,ham,"Nah I don't think he goes to usf, he lives aro...",61,1,nah i dont think he goes to usf he lives aroun...
6,spam,FreeMsg Hey there darling it's been 3 week's n...,148,0,freemsg hey there darling its been 3 weeks now...
7,ham,Even my brother is not like to speak with me. ...,77,1,even my brother is not like to speak with me t...
8,ham,As per your request 'Melle Melle (Oru Minnamin...,160,1,as per your request melle melle oru minnaminun...


Now we need to convert each of those messages into a vector the SciKit Learn's algorithm models can work with and machine learning model which we will gonig to use can understand.

In [46]:
# pip install nltk

In [38]:
import nltk
from nltk.corpus import stopwords
#Create stopword list:
nltk.download('stopwords')
print(stopwords.words('english'), len(stopwords.words('english')))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [52]:
# Countvectorizer is a method to convert text to numerical data.
# Initialize the object for countvectorizer
CV = CountVectorizer(stop_words="english")

[Stopwords are the words in any language which does not add much meaning to a sentence. They are the words which are very common in text documents such as a, an, the, you, your, etc. The Stop Words highly appear in text documents. However, they are not being helpful for text analysis in many of the cases, So it is better to remove from the text. We can focus on the important words if stop words have removed.]

In [53]:
CV.fit(df['text_clean'])

CountVectorizer(stop_words='english')

In [54]:
X=CV.transform(df['text_clean']).toarray()

In [57]:
df.head()

,Class,Message,Length,Target,text_clean
1,ham,"Go until jurong point, crazy.. Available only ...",111,1,go until jurong point crazy available only in ...
2,ham,Ok lar... Joking wif u oni...,29,1,ok lar joking wif u oni
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,free entry in 2 a wkly comp to win fa cup fina...
4,ham,U dun say so early hor... U c already then say...,49,1,u dun say so early hor u c already then say
5,ham,"Nah I don't think he goes to usf, he lives aro...",61,1,nah i dont think he goes to usf he lives aroun...


In [59]:
X, len(X)

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=int64),
 5572)

In [60]:
x=CV.transform(df['Message'].astype(str)).toarray()

In [61]:
df.head()

,Class,Message,Length,Target,text_clean
1,ham,"Go until jurong point, crazy.. Available only ...",111,1,go until jurong point crazy available only in ...
2,ham,Ok lar... Joking wif u oni...,29,1,ok lar joking wif u oni
3,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,0,free entry in 2 a wkly comp to win fa cup fina...
4,ham,U dun say so early hor... U c already then say...,49,1,u dun say so early hor u c already then say
5,ham,"Nah I don't think he goes to usf, he lives aro...",61,1,nah i dont think he goes to usf he lives aroun...


In [62]:
# Splitting x and y

XSet = df['text_clean'].values
ySet = df['Target'].values
ySet

array([1, 1, 0, ..., 1, 1, 1], dtype=int64)

In [63]:
# Datatype for y is object. lets convert it into int
ySet = ySet.astype('int')
ySet

array([1, 1, 0, ..., 1, 1, 1])

In [64]:
XSet

array(['go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat',
       'ok lar joking wif u oni',
       'free entry in 2 a wkly comp to win fa cup final tkts 21st may 2005 text fa to 87121 to receive entry questionstd txt ratetcs apply 08452810075over18s',
       ..., 'pity  was in mood for that soany other suggestions',
       'the guy did some bitching but i acted like id be interested in buying something else next week and he gave it to us for free',
       'rofl its true to its name'], dtype=object)

### Splitting Train and Test Data

In [65]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [66]:
y=df['Target']
y

1       1
2       1
3       0
4       1
5       1
       ..
5568    0
5569    1
5570    1
5571    1
5572    1
Name: Target, Length: 5572, dtype: int64

In [67]:
XSet_train,XSet_test,ySet_train,ySet_test = train_test_split(XSet,ySet,test_size=0.2, random_state=42)

In [68]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25, random_state=25)

In [69]:
XSet_train_CV = CV.fit_transform(XSet_train)
XSet_train_CV

<4457x8121 sparse matrix of type '<class 'numpy.int64'>'
	with 34258 stored elements in Compressed Sparse Row format>

In [70]:
XSet_test.shape

(1115,)

### Training a model

With messages represented as vectors, we can finally train our spam/ham classifier. Now we can actually use almost any sort of classification algorithms. For a variety of reasons, the Naive Bayes classifier algorithm is a good choice.

In [71]:
# Initialising the model
NB = MultinomialNB()

In [72]:
NB.fit(X_train,y_train)

MultinomialNB()

In [73]:
# feed data to the model
NB.fit(XSet_train_CV,ySet_train)

MultinomialNB()

In [74]:
# Let's test CV on our test data
XSet_test_CV = CV.transform(XSet_test)

In [75]:
# y_pred=NB.predict(x_test)
y_pred=NB.predict(XSet_test_CV)

In [76]:
y_pred

array([1, 1, 0, ..., 1, 1, 0])

In [77]:
# prediction for xSet_test_CV

ySet_predict = NB.predict(XSet_test_CV)
ySet_predict

array([1, 1, 0, ..., 1, 1, 0])

In [78]:
# Checking accuracy

accuracyScore = accuracy_score(ySet_test,ySet_predict)*100

print("Prediction Accuracy :",accuracyScore)

Prediction Accuracy : 98.20627802690582


### SpamClassificationApplication

In [79]:
msg = input("Enter Message: ") # to get the input message
msgInput = CV.transform([msg]) #
predict = NB.predict(msgInput)
if(predict[0]==0):
    print("------------------------MESSAGE-SENT-[CHECK-SPAM-FOLDER]---------------------------")
else:
    print("---------------------------MESSAGE-SENT-[CHECK-INBOX]------------------------------")

Enter Message: cgfyghb
---------------------------MESSAGE-SENT-[CHECK-INBOX]------------------------------


## BAG OF WORDS

We cannot pass text directly to train our models in Natural Language Processing, thus we need to convert it into numbers, which machine can understand and can perform the required modelling on it

In [73]:
# Let's understand it with a simple example

In [74]:
# creating a list of sentences
documents = ["Dog bites man.", "Man bites dog.", "Dog eats meat.", "Man eats food."]

# Changing the text to lower case and remove the full stop from text
processed_docs = [doc.lower().replace(".","") for doc in documents]
processed_docs[3]

'man eats food'

In [75]:
# corpus is the collection of text
#look at the documents list
print("Our corpus: ", processed_docs)


# Initialise the object for CountVectorizer
count_vect = CountVectorizer()

#Build a BOW representation for the corpus
bow_rep = count_vect.fit_transform(processed_docs)
# print(bow_rep)

#Look at the vocabulary mapping
print("Our vocabulary: ", count_vect.vocabulary_)

#see the BOW rep for first 2 documents
print("BoW representation for 'dog bites man': ", bow_rep[0].toarray())
print("BoW representation for 'man bites dog: ",bow_rep[1].toarray())

#Get the representation using this vocabulary, for a new text
temp = count_vect.transform(["dog and dog are friends"])
print("Bow representation for 'dog and dog are friends':", temp.toarray())

Our corpus:  ['dog bites man', 'man bites dog', 'dog eats meat', 'man eats food']
Our vocabulary:  {'dog': 1, 'bites': 0, 'man': 4, 'eats': 2, 'meat': 5, 'food': 3}
BoW representation for 'dog bites man':  [[1 1 0 0 1 0]]
BoW representation for 'man bites dog:  [[1 1 0 0 1 0]]
Bow representation for 'dog and dog are friends': [[0 2 0 0 0 0]]


## TF-IDF

In **BOW approach** we saw so far, all the words in the text are treated equally important. There is no notion of some words in the document being more important than others. TF-IDF addresses this issue. It aims to quantify the importance of a given word relative to other words in the document and in the


<font color=darkviolet>  **Term Frequency (tf)** </font>
TF: Term Frequency, which measures how frequently a term occurs in a document. Since every document is different in length, it is possible that a term would appear much more times in long documents than shorter ones. Thus, the term frequency is often divided by the document length (aka. the total number of terms in the document) as a way of normalization:

TF(t) = (Number of times term 't' appears in a document) / (Total number of terms in the document).



<font color=darkviolet>  **Inverse Document Frequency (idf)** </font>
              It measures how important a term is. While computing TF, all terms are considered equally important. However it is known that certain terms, such as "is", "of", and "that", may appear a lot of times but have little importance. Thus we need to weigh down the frequent terms while scale up the rare ones, by computing the following:

IDF(t) = log_e(Total number of documents / Number of documents with term t in it).corpus. It was commonly used representation scheme for information retrieval systems, for extracting relevant documents from a corpus for given text query.



__Let's see an example:__

Consider a document containing 100 words wherein the word cat appears 3 times.

The term frequency (i.e., tf) for cat is then (3 / 100) = 0.03.

Now, assume we have 10 million documents and the word cat appears in one thousand of these.

Then, the inverse document frequency (i.e., idf) is calculated as log(10,000,000 / 1,000) = 4.

Thus, the Tf-idf weight is the product of these quantities: 0.03 * 4 = 0.12

In [76]:
# Splitting x and y

X = df['text_clean'].values
y = df['Target'].values
y

array([1, 1, 0, ..., 1, 1, 1], dtype=int64)

In [77]:
# Datatype for y is object. lets convert it into int
y = y.astype('int')
y

array([1, 1, 0, ..., 1, 1, 1])

In [78]:
type(X)

numpy.ndarray

In [79]:
## text preprocessing and feature vectorizer
# To extract features from a document of words, we import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


tf=TfidfVectorizer() ## object creation
X=tf.fit_transform(X) ## fitting and transforming the data into vectors


In [80]:
X.shape

(5572, 9489)

In [81]:
## print feature names selected from the raw documents
# tf.get_feature_names()
tf.get_feature_names_out()

array(['008704050406', '0089my', '0121', ..., 'ûïharry', 'ûò', 'ûówell'],
      dtype=object)

In [82]:
## number of features created
len(tf.get_feature_names_out())

9489

In [83]:
X

<5572x9489 sparse matrix of type '<class 'numpy.float64'>'
	with 72459 stored elements in Compressed Sparse Row format>

In [84]:
## getting the feature vectors
X=X.toarray()

In [85]:
## Creating training and testing
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=6)

In [86]:
## Model creation
from sklearn.naive_bayes import BernoulliNB

## model object creation
nb=BernoulliNB(alpha=0.01)

## fitting the model
nb.fit(X_train,y_train)

## getting the prediction
y_hat=nb.predict(X_test)

In [87]:
y_hat

array([1, 1, 1, ..., 1, 1, 1])

In [88]:
## Evaluating the model
from sklearn.metrics import classification_report,confusion_matrix

In [89]:
print(classification_report(y_test,y_hat))

              precision    recall  f1-score   support

           0       0.98      0.93      0.96       201
           1       0.99      1.00      0.99      1192

    accuracy                           0.99      1393
   macro avg       0.99      0.96      0.97      1393
weighted avg       0.99      0.99      0.99      1393



In [90]:
## confusion matrix
pd.crosstab(y_test,y_hat)

col_0,0,1
row_0,,
0,187,14
1,3,1189


### Pros of Naive Bayes

- Naive Bayes Algorithm is a fast, highly scalable algorithm
- Naive Bayes can be classified for both binary classification and multi class classification. It provides different types of Naive Bayes Algorithms like GaussianNB, MultinominalNB, BernoulliNB.
- It is simple algorithm that depends on doing a bunch of count.
- Great choice for text classification problems. it's a popular choice for spam email classification.
- It can be easily trained on small datasets.
- Naive Bayes can handle misssing data, as they ignored when a probabilty is calculated for a class value.


### Cons of Naive Bayes

- It considers all the features to be unrelated, so it cannot learn the relationship between features. This limits the applicability of this algorithm in real-world use cases.
- Naive Bayes can learn individual featutre importance but can't determine the relationship among features.

## Application of Naive Bayes

##### Text classification / spam filtering / Sentiment analysis:
 - Naive Bayes classifiers mostly used in text classification
 - News article classification SPORTS, TECHNOLOGY etc.
 - Spam or Ham: Naive Bayes is the most popular method for mail filtering
 - Sentiment analysis focuses on identifying whether the customers think positively or negatively about a certain topic (product or service).


##### Recommendation System:
- Naive Bayes classifier and Collabrative filtering together buids a recommendation system that uses machine learning and data mining techniques to filter unseen information and predict whether a user would like a given resource or not.



### 3 Types of Naive Bayes in Scikit Learn

__Gaussian__

- It is used in classification and it assumes that features follow a normal distribution.

__Multinominal__
- It is used for discrete counts. For eg., let's say we have a text cLassification problem. Here we consider Bernoulli trails which is one step further and instead of "word occuring in the document", we have "count how often word occurs in the document" you can think of it as "number of times outcome number_x is observed over n trails".

__Bernoulli__
- The binomial model is useful if your feature vectors are binary (ie., Zeroes and One). One application would be text classification with 'bag of words' model where the 1s and 0s are "words occur in the document" and "word does not occur in the document" respectively.